In [3]:
from datasets.arrow_writer import ArrowWriter
from tqdm import tqdm
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import soundfile as sf
import string
import csv
import json

DATASET_DIRS = [Path("../data/tts_dataset"), Path("../data/stt_dataset"), Path("../data/bonus_dataset")]
DATASET_DIR = DATASET_DIRS[1]
PROJECT_NAME = "f5_dataset"
OUT_ROOT = Path(PROJECT_NAME)
SPLITS = ["train", "val", "test"]

@dataclass
class Item:
    audio_path: str
    text: str
    duration: float

def wav_duration_seconds(path: Path) -> float:
    info = sf.info(str(path))
    return info.frames / float(info.samplerate)

def read_split(split: str) -> list[Item]:
    root = DATASET_DIR / split
    meta = root / "metadata.csv"

    items = []
    with meta.open("r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        header = next(reader, None)
        if header and header[0] != "audio_file":
            f.seek(0)
            reader = csv.reader(f, delimiter="\t")

        for row in reader:
            rel_audio, text = row[0], row[1] if len(row) > 1 else ""
            audio_path = (root / 'wavs' / rel_audio).resolve()
            dur = wav_duration_seconds(audio_path)
            items.append(Item(str(audio_path), text, dur))
    return items

def read_split_all_dirs(split: str) -> list[Item]:
    items = []
    for dataset_dir in DATASET_DIRS:
        root = dataset_dir / split
        meta = root / "metadata.csv"
        if not meta.exists():
            print(f"[!] Пропускаем {meta} — нет файла metadata.csv")
            continue

        with meta.open("r", encoding="utf-8") as f:
            reader = csv.reader(f, delimiter="\t")
            header = next(reader, None)
            if header and header[0] != "audio_file":
                f.seek(0)
                reader = csv.reader(f, delimiter="\t")

            for row in reader:
                rel_audio, text = row[0], row[1] if len(row) > 1 else ""
                audio_path = (root / 'wavs' / rel_audio).resolve()
                if not audio_path.exists():
                    print(f"[!] Пропускаем {audio_path} — нет файла")
                    continue
                dur = wav_duration_seconds(audio_path)
                items.append(Item(str(audio_path), text, dur))
    return items

def save_split_arrow(split: str, items: list[Item]) -> Path:
    """Сохраняет split в формате raw.arrow + duration.json, как в prepare_csv_wavs.py"""
    out_dir = OUT_ROOT / split
    out_dir.mkdir(parents=True, exist_ok=True)

    raw_arrow_path = out_dir / "raw.arrow"
    with ArrowWriter(path=raw_arrow_path.as_posix()) as writer:
        for it in tqdm(items, desc=f"Writing {split}/raw.arrow"):
            writer.write({
                "audio_path": it.audio_path,
                "text": it.text,
                "duration": it.duration
            })
        writer.finalize()

    # Сохраняем duration.json
    with (out_dir / "duration.json").open("w", encoding="utf-8") as f:
        json.dump({"duration": [it.duration for it in items]}, f, ensure_ascii=False, indent=2)

    return out_dir

def build_vocab(items: list[Item]) -> list[str]:
    freq = Counter()
    for it in items:
        freq.update(it.text)
    drop_chars = {"\n", "\r", "\t"}
    train_chars = set(freq.keys()) - drop_chars

    extra_chars = (
        [chr(i) for i in range(ord('А'), ord('Я')+1)] +
        [chr(i) for i in range(ord('а'), ord('я')+1)] +
        list(string.ascii_uppercase + string.ascii_lowercase) +
        list(string.digits) +
        list(".,!?;:()[]{}«»\"'…–-") +
        [" "]
    )

    vocab_set = train_chars.union(extra_chars)
    return sorted(vocab_set)



OUT_ROOT.mkdir(parents=True, exist_ok=True)

all_items_by_split = {}
for split in SPLITS:
    items = read_split_all_dirs(split)
    save_split_arrow(split, items)
    all_items_by_split[split] = items
    print(f"[OK] {split}: {len(items)} примеров")

vocab_sorted = build_vocab(all_items_by_split["train"])
vocab_path = OUT_ROOT / "vocab.txt"
with vocab_path.open("w", encoding="utf-8", newline="\n") as f:
    f.writelines(ch + "\n" for ch in vocab_sorted)

print(f"[OK] vocab: {vocab_path} (символов: {len(vocab_sorted)})")
print(f"\nГотово. Данные лежат в: {OUT_ROOT}/{{train,val,test}}/raw + duration.json")


Writing train/raw.arrow: 100%|██████████| 34138/34138 [00:00<00:00, 718920.05it/s]


[OK] train: 34138 примеров


Writing val/raw.arrow: 100%|██████████| 300/300 [00:00<00:00, 3106891.85it/s]


[OK] val: 300 примеров


Writing test/raw.arrow: 100%|██████████| 3795/3795 [00:00<00:00, 756206.17it/s]

[OK] test: 3795 примеров
[OK] vocab: f5_dataset/vocab.txt (символов: 160)

Готово. Данные лежат в: f5_dataset/{train,val,test}/raw + duration.json
